In [5]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import warnings

from tqdm import tqdm
import time
warnings.filterwarnings('ignore')

# ----------------------------
# Logging
# ----------------------------
import logging

logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s %(levelname)s %(message)s",
    handlers=[
        logging.FileHandler("debug.log", encoding="utf-8"),  # 파일: 무제한
        logging.StreamHandler()                              # 노트북 셀: 요약만
    ]
)
# Jupyter notebook에서 출력 제한 해제
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
# 완전 수정된 버전

class ManualMultiOutputRegressor:
    """가중치를 확실히 지원하는 MultiOutput 래퍼"""
    def __init__(self, base_estimator):
        self.base_estimator = base_estimator
        self.estimators_ = []
    
    def fit(self, X, y, sample_weight=None):
        from sklearn.base import clone
        self.estimators_ = []
        
        print(f"MultiOutput 학습: {y.shape[1]}개 타겟")
        if sample_weight is not None:
            print(f"가중치 통계: 평균={sample_weight.mean():.4f}, "
                 f"최소={sample_weight.min():.4f}, 최대={sample_weight.max():.4f}")
        
        for i in range(y.shape[1]):
            estimator = clone(self.base_estimator)
            if sample_weight is not None:
                estimator.fit(X, y[:, i], sample_weight=sample_weight)
            else:
                estimator.fit(X, y[:, i])
            self.estimators_.append(estimator)
        return self
    
    def predict(self, X):
        predictions = []
        for estimator in self.estimators_:
            pred = estimator.predict(X)
            predictions.append(pred)
        return np.column_stack(predictions)

class ClusterBasedForecastingModel:
    def __init__(self):
        self.menu_clusters = {}
        self.cluster_models = {}
        self.cluster_features = {}
        self.store_patterns = {}
        
        # 가중치 전처리기 통합
        from weight_processor import WeightedTrainingPreprocessor
        self.weight_preprocessor = WeightedTrainingPreprocessor(
            min_consecutive_zeros=100,
            closure_weight=0.1,      # 휴업 기간 가중치
            pre_launch_weight=0.05   # 출시 전 기간 가중치
        )

    def classify_menu_stability(self, train_df):
        """메뉴별 안정성 분류"""
        
        menu_stability = {}
        
        for (store, menu), group in train_df.groupby(['store', 'menu']):
            if len(group) < 30:  # 데이터 부족시 제외
                continue
                
            zero_ratio = (group['sales'] == 0).mean()
            total_sales = group['sales'].sum()
            
            # 안정성 분류 기준
            if zero_ratio < 0.1:
                stability = 'very_stable'
            elif zero_ratio < 0.3:
                stability = 'stable'  
            elif zero_ratio < 0.6:
                stability = 'moderate'
            else:
                stability = 'unstable'
                
            menu_stability[(store, menu)] = {
                'stability': stability,
                'zero_ratio': zero_ratio,
                'total_sales': total_sales,
                'avg_sales': group['sales'].mean(),
                'volatility': group['sales'].std() / (group['sales'].mean() + 1e-8)
            }
        
        self.menu_stability_map = menu_stability
        return menu_stability
    
    def analyze_and_cluster_menus(self, train_df):
        """메뉴 클러스터링 및 클러스터별 특성 분석"""
        
        menu_features = []
        menu_names = []
        
        logging.debug("메뉴 클러스터링 시작...")
        
        # 1. 각 메뉴별 특성 추출
        grouped = train_df.groupby(['store', 'menu'])
        for (store, menu), group in tqdm(grouped, desc="메뉴 특성 추출"):
            if len(group) < 20:  # 충분한 데이터가 있는 메뉴만
                continue
                
            # 메뉴별 특성 벡터 생성
            features = {
                'avg_sales': group['sales'].mean(),
                'std_sales': group['sales'].std(),
                'zero_ratio': (group['sales'] == 0).mean(),
                'max_sales': group['sales'].max(),
                'cv': group['sales'].std() / (group['sales'].mean() + 1e-8),  # 변동계수
                'weekend_boost': group[group['is_weekend']]['sales'].mean() / (group['sales'].mean() + 1e-8),
                'seasonality_strength': self._calculate_seasonality(group),
                'trend_strength': self._calculate_trend(group['sales'].values),
            }
            
            # 메뉴명 기반 특성
            menu_lower = str(menu).lower()
            features.update({
                'is_main_dish': int(any(x in menu_lower for x in ['불고기', '갈비', '찌개', '국밥', '정식'])),
                'is_drink': int(any(x in menu_lower for x in ['콜라', '맥주', '소주', '커피', '음료', '차'])),
                'is_premium': int(any(x in menu_lower for x in ['한우', 'aus', '프리미엄'])),
                'is_group': int('단체' in menu_lower),
                'is_brunch': int('브런치' in menu_lower),
            })
            
            menu_features.append(list(features.values()))
            menu_names.append((store, menu))
        
        if len(menu_features) < 5:
            logging.debug("클러스터링에 충분한 메뉴가 없음")
            return {}
        
        # 2. K-means 클러스터링
        from sklearn.cluster import KMeans
        from sklearn.preprocessing import StandardScaler
        
        scaler = StandardScaler()
        features_scaled = scaler.fit_transform(menu_features)
        
        # 최적 클러스터 수 결정
        n_clusters = min(6, max(3, len(menu_features) // 8))
        logging.debug(f"클러스터 수: {n_clusters}")
        
        kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
        cluster_labels = kmeans.fit_predict(features_scaled)
        
        # 3. 클러스터별 메뉴 그룹화 및 특성 분석
        cluster_groups = {}
        cluster_characteristics = {}
        
        for cluster_id in tqdm(range(n_clusters), desc="클러스터 특성 분석"):
            cluster_mask = cluster_labels == cluster_id
            cluster_menus = [menu_names[i] for i in range(len(menu_names)) if cluster_mask[i]]
            cluster_groups[cluster_id] = cluster_menus
            
            # 클러스터 특성 계산
            cluster_data_list = []
            for store, menu in cluster_menus:
                menu_data = train_df[(train_df['store'] == store) & (train_df['menu'] == menu)]
                cluster_data_list.append(menu_data)
            
            if cluster_data_list:
                cluster_combined = pd.concat(cluster_data_list, ignore_index=True)
                
                cluster_characteristics[cluster_id] = {
                    'avg_sales': cluster_combined['sales'].mean(),
                    'zero_ratio': (cluster_combined['sales'] == 0).mean(),
                    'volatility': cluster_combined['sales'].std() / (cluster_combined['sales'].mean() + 1e-8),
                    'weekend_effect': cluster_combined[cluster_combined['is_weekend']]['sales'].mean() / 
                                    (cluster_combined[~cluster_combined['is_weekend']]['sales'].mean() + 1e-8),
                    'menu_count': len(cluster_menus),
                    'dominant_type': self._get_dominant_menu_type(cluster_menus)
                }
        
        self.menu_clusters = cluster_groups
        self.cluster_features = cluster_characteristics
        
        # 클러스터 정보 출력
        for cluster_id, info in cluster_characteristics.items():
            logging.debug(f"클러스터 {cluster_id}: {info['menu_count']}개 메뉴, "
                  f"평균매출 {info['avg_sales']:.1f}, 타입: {info['dominant_type']}")
        
        return cluster_groups
    
    def _calculate_seasonality(self, group):
        """계절성 강도 계산"""
        if 'month' not in group.columns:
            return 0
        monthly_avg = group.groupby('month')['sales'].mean()
        if len(monthly_avg) < 2:
            return 0
        return monthly_avg.std() / (monthly_avg.mean() + 1e-8)
    
    def _calculate_trend(self, sales):
        """트렌드 강도 계산"""
        if len(sales) < 2:
            return 0
        x = np.arange(len(sales))
        return abs(np.polyfit(x, sales, 1)[0])
    
    def _get_dominant_menu_type(self, cluster_menus):
        """클러스터의 주요 메뉴 타입 결정"""
        type_counts = {'main': 0, 'drink': 0, 'premium': 0, 'group': 0, 'brunch': 0, 'other': 0}
        
        for store, menu in cluster_menus:
            menu_lower = str(menu).lower()
            if any(x in menu_lower for x in ['불고기', '갈비', '찌개', '국밥', '정식']):
                type_counts['main'] += 1
            elif any(x in menu_lower for x in ['콜라', '맥주', '소주', '커피', '음료']):
                type_counts['drink'] += 1
            elif any(x in menu_lower for x in ['한우', 'aus', '프리미엄']):
                type_counts['premium'] += 1
            elif '단체' in menu_lower:
                type_counts['group'] += 1
            elif '브런치' in menu_lower:
                type_counts['brunch'] += 1
            else:
                type_counts['other'] += 1
        
        return max(type_counts, key=type_counts.get)
    
    def get_menu_cluster(self, store, menu):
        """특정 메뉴의 클러스터 ID 반환"""
        for cluster_id, menus in self.menu_clusters.items():
            if (store, menu) in menus:
                return cluster_id
        return -1  # 클러스터에 없음
    
    def create_cluster_features(self, df, sample_weights, mode='train'):
        """클러스터 정보를 활용한 피처 생성"""
        
        sequences = []
        targets = []
        metadata = []
        feature_weights = []  # 시퀀스별 가중치

        weight_dict = {}
        for idx, weight in enumerate(sample_weights):
            weight_dict[idx] = weight
        
        for (store, menu), group in df.groupby(['store', 'menu']):
            group = group.sort_values('date').reset_index(drop=True)
            
            min_length = 28 + (7 if mode == 'train' else 0)
            if len(group) < min_length:
                continue
            
            # 클러스터 정보 가져오기
            cluster_id = self.get_menu_cluster(store, menu)
            cluster_info = self.cluster_features.get(cluster_id, {})
            
            if mode == 'predict':
                seq_data = group.tail(28)
                features = self._create_cluster_based_features(seq_data, store, menu, cluster_id, cluster_info)
                sequences.append(features)
                metadata.append({'store': store, 'menu': menu, 'cluster': cluster_id})
                feature_weights.append(1.0)  # 예측 시에는 가중치 1.0
                
            else:
                for i in range(len(group) - min_length + 1):
                    seq_data = group.iloc[i:i+28]
                    target_data = group.iloc[i+28:i+35]
                    
                    features = self._create_cluster_based_features(seq_data, store, menu, cluster_id, cluster_info)
                    target = target_data['sales'].values

                    # 시퀀스 가중치 계산 (28일 구간의 평균 가중치)
                    seq_weight = np.mean([weight_dict.get(group.index[i+j], 1.0) for j in range(28)])
                    
                    
                    sequences.append(features)
                    targets.append(target)
                    metadata.append({'store': store, 'menu': menu, 'cluster': cluster_id})
                    feature_weights.append(seq_weight)
        
        X = np.array(sequences) if sequences else np.empty((0, 60))
        y = np.array(targets) if targets else np.empty((0, 7))

        weights = np.array(feature_weights) if feature_weights else np.empty((0,))

        
        return X, y, metadata, weights
    
    def _create_cluster_based_features(self, seq_data, store, menu, cluster_id, cluster_info):
        """클러스터 정보를 활용한 피처 생성"""
        
        sales = seq_data['sales'].values
        
        # 1. 기본 통계 피처
        basic_features = [
            np.mean(sales), np.median(sales), np.std(sales),
            np.min(sales), np.max(sales),
            np.mean(sales[-7:]), np.mean(sales[-14:]),
            sales[-1] if len(sales) > 0 else 0,
            (sales == 0).mean(),
            self._calculate_trend(sales),
        ]
        
        # 2. 클러스터 기반 피처 (핵심!)
        cluster_features = [
            cluster_id if cluster_id != -1 else 0,  # 클러스터 ID
            cluster_info.get('avg_sales', 0) / (np.mean(sales) + 1e-8),  # 클러스터 평균 대비 비율
            cluster_info.get('zero_ratio', 0),  # 클러스터 0매출 비율
            cluster_info.get('volatility', 0),  # 클러스터 변동성
            cluster_info.get('weekend_effect', 1),  # 클러스터 주말효과
        ]
        
        # 3. 클러스터별 상대적 성과
        cluster_avg = cluster_info.get('avg_sales', np.mean(sales))
        relative_performance = [
            np.mean(sales) / (cluster_avg + 1e-8),  # 현재 성과 vs 클러스터 평균
            (sales > cluster_avg).mean(),  # 클러스터 평균 초과 비율
            np.std(sales) / (cluster_info.get('volatility', 1) * cluster_avg + 1e-8),  # 상대적 변동성
        ]
        
        # 4. 클러스터 타입별 피처
        cluster_type = cluster_info.get('dominant_type', 'other')
        type_features = [
            int(cluster_type == 'main'),
            int(cluster_type == 'drink'), 
            int(cluster_type == 'premium'),
            int(cluster_type == 'group'),
            int(cluster_type == 'brunch'),
        ]
        
        # 5. 시간 기반 피처
        time_features = [
            seq_data['month'].iloc[-1],
            seq_data['day_of_week'].iloc[-1],
            seq_data['is_weekend'].sum(),
            np.sin(2 * np.pi * seq_data['month'].iloc[-1] / 12),
            np.cos(2 * np.pi * seq_data['month'].iloc[-1] / 12),
            np.sin(2 * np.pi * seq_data['day_of_week'].iloc[-1] / 7),
            np.cos(2 * np.pi * seq_data['day_of_week'].iloc[-1] / 7),
        ]
        
        # 6. 업장 기반 피처
        store_features = self._get_store_features(store)
        
        # 7. 라그 피처
        lag_features = [
            sales[-1] if len(sales) >= 1 else 0,
            sales[-7] if len(sales) >= 7 else 0,
            sales[-14] if len(sales) >= 14 else 0,
        ]

         # 7. 가중치 관련 피처 (3개) - 핵심 추가!
        weight_features = [
            seq_data.get('is_closure_period', pd.Series([0] * len(seq_data))).iloc[-1],
            seq_data.get('is_pre_launch', pd.Series([0] * len(seq_data))).iloc[-1], 
            seq_data.get('store_special_status', pd.Series([0] * len(seq_data))).iloc[-1],
        ]
        # 모든 피처 결합
        all_features = (basic_features + cluster_features + relative_performance + 
                       type_features + time_features + store_features + lag_features + weight_features)
        
        # NaN 처리 및 크기 조정
        all_features = [0.0 if pd.isna(x) or np.isinf(x) else float(x) for x in all_features]
        
        # 60개 피처로 맞추기
        while len(all_features) < 60:
            all_features.append(0.0)
        all_features = all_features[:60]
        
        return all_features
    
    def _get_store_features(self, store):
        """업장 원-핫 인코딩"""
        stores = ['담하', '미라시아', '포레스트릿', '카페테리아', '화담숲주막', 
                 '화담숲카페', '느티나무 셀프BBQ', '연회장', '라그로타']
        return [int(store == s) for s in stores]

    def fit(self, train_df):
        """수정된 가중치 기반 학습"""
        logging.debug("클러스터 기반 모델 학습 시작...")
        
        # 1. 가중치 전처리 수행
        processed_df, sample_weights = self.weight_preprocessor.process_training_data(train_df)
        
        # 가중치 통계 확인
        logging.debug(f"가중치 통계:")
        logging.debug(f"  평균: {sample_weights.mean():.4f}")
        logging.debug(f"  최소: {sample_weights.min():.4f}")
        logging.debug(f"  최대: {sample_weights.max():.4f}")
        logging.debug(f"  0.5 미만 비율: {(sample_weights < 0.5).mean():.2%}")
        
        # 2. 메뉴 클러스터링
        self.analyze_and_cluster_menus(processed_df)
        
        # 3. 클러스터 기반 피처 생성
        X, y, metadata, weights = self.create_cluster_features(processed_df, sample_weights, mode='train')
        
        if len(X) == 0:
            logging.debug("학습 데이터 부족")
            return
        
        # 시퀀스 가중치 통계 확인
        logging.debug(f"시퀀스 가중치 통계:")
        logging.debug(f"  평균: {weights.mean():.4f}")
        logging.debug(f"  최소: {weights.min():.4f}")
        logging.debug(f"  최대: {weights.max():.4f}")
        logging.debug(f"  저가중치({self.weight_preprocessor.closure_weight} 이하) 비율: "
             f"{(weights <= self.weight_preprocessor.closure_weight).mean():.1%}")
        
        logging.debug(f"클러스터 피처: {X.shape}, 타겟: {y.shape}, 가중치: {len(weights)}")
        
        # 4. 가중치 기반 모델 학습
        from sklearn.ensemble import RandomForestClassifier
        from lightgbm import LGBMRegressor
        from xgboost import XGBRegressor
        
        # 매출 여부 분류기 (가중치 적용)
        has_sales = (y > 0).any(axis=1)
        self.zero_classifier = RandomForestClassifier(
            n_estimators=200, max_depth=12, random_state=42, class_weight='balanced'
        )
        print("분류기 가중치 적용 중...")
        self.zero_classifier.fit(X, has_sales, sample_weight=weights)
        
        sales_mask = has_sales
        if sales_mask.sum() > 20:
            
            # 전체 모델 (수정된 MultiOutputRegressor 사용)
            print("전체 모델 학습 시작...")
            self.global_model = ManualMultiOutputRegressor(
                LGBMRegressor(n_estimators=600, max_depth=8, learning_rate=0.05, 
                            random_state=42, verbosity=-1)
            )
            
            with tqdm(total=100, desc="전체 모델 학습") as pbar:
                self.global_model.fit(X[sales_mask], y[sales_mask], sample_weight=weights[sales_mask])
                pbar.update(100)
            
            # 클러스터별 모델 학습 (수정됨)
            logging.debug("클러스터별 전용 모델 학습 중...")
            self.cluster_models = {}
            
            valid_clusters = []
            for cluster_id in self.menu_clusters.keys():
                cluster_mask = np.array([meta['cluster'] == cluster_id for meta in metadata])
                cluster_sales_mask = sales_mask & cluster_mask
                if cluster_sales_mask.sum() > 10:
                    valid_clusters.append((cluster_id, cluster_sales_mask))

            for cluster_id, cluster_sales_mask in tqdm(valid_clusters, desc="클러스터 모델"):
                
                cluster_model = ManualMultiOutputRegressor(
                    XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.08,
                                random_state=42, verbosity=0)
                )
                
                # 가중치와 함께 학습 - 확실히 적용!
                cluster_weights = weights[cluster_sales_mask]
                cluster_model.fit(X[cluster_sales_mask], y[cluster_sales_mask], 
                                sample_weight=cluster_weights)
                self.cluster_models[cluster_id] = cluster_model
                
                # 클러스터별 가중치 분포 확인
                cluster_info = self.cluster_features[cluster_id]
                tqdm.write(f"클러스터 {cluster_id} 완료: {cluster_info['dominant_type']}, "
                        f"{cluster_sales_mask.sum()}개 샘플, 평균 가중치: {cluster_weights.mean():.3f}")
        
        logging.debug(f"전체 모델 + {len(self.cluster_models)}개 클러스터 모델 학습 완료")
    
    def _add_weight_features_to_test(self, test_df):
        """테스트 데이터에 가중치 관련 피처 추가 (예측 전용)"""
        processed_df = test_df.copy()
        
        # 휴업 기간 피처
        processed_df['is_closure_period'] = (
            (processed_df['store'].isin(['화담숲주막', '화담숲카페'])) & 
            (processed_df['month'].isin([12, 1, 2]))
        ).astype(int)
        
        # 출시 전 기간 피처 (학습된 정보 활용)
        processed_df['is_pre_launch'] = 0
        if hasattr(self.weight_preprocessor, 'launch_analysis'):
            for (store, menu), launch_info in self.weight_preprocessor.launch_analysis.items():
                mask = (processed_df['store'] == store) & (processed_df['menu'] == menu) & \
                       (processed_df['date'] < launch_info['first_sale_date'])
                processed_df.loc[mask, 'is_pre_launch'] = 1
        
        # 특수 상태 피처
        processed_df['store_special_status'] = 0
        processed_df.loc[processed_df['is_closure_period'] == 1, 'store_special_status'] = 1
        processed_df.loc[processed_df['is_pre_launch'] == 1, 'store_special_status'] = 2
        
        return processed_df
    
    def predict(self, test_df):
        """수정된 예측 함수"""
        if not hasattr(self, 'global_model'):
            logging.debug("모델이 학습되지 않음")
            return np.zeros((len(test_df), 7)), []
        
        # 테스트 데이터 전처리 (훈련용 함수 사용 안함!)
        processed_test_df = self._add_weight_features_to_test(test_df)
        dummy_weights = np.ones(len(processed_test_df))
        
        X, _, metadata, _ = self.create_cluster_features(processed_test_df, dummy_weights, mode='predict')
        
        if len(X) == 0:
            return np.zeros((0, 7)), []
        
        logging.debug("매출 여부 예측 중...")
        with tqdm(total=100, desc="분류 예측") as pbar:
            has_sales_prob = self.zero_classifier.predict_proba(X)[:, 1]
            pbar.update(100)
        
        # 예측 수행
        predictions = np.zeros((len(X), 7))
        
        for i in tqdm(range(len(X)), desc="예측 진행"):
            cluster_id = metadata[i]['cluster']
            
            if cluster_id in self.cluster_models:
                cluster_pred = self.cluster_models[cluster_id].predict(X[i:i+1])
                global_pred = self.global_model.predict(X[i:i+1])
                
                cluster_weight = 0.7
                predictions[i] = cluster_weight * cluster_pred[0] + (1-cluster_weight) * global_pred[0]
            else:
                predictions[i] = self.global_model.predict(X[i:i+1])[0]
        
        # Zero-inflation 적용
        threshold = 0.3
        zero_mask = has_sales_prob < threshold
        predictions[zero_mask] = 0
        
        # 후처리
        predictions = np.maximum(predictions, 1)
        
        return predictions, metadata

# 가중치 효과 검증 함수
def validate_weight_effect(model, train_df):
    """가중치 효과 검증"""
    print("\n=== 가중치 효과 검증 ===")
    
    processed_df, sample_weights = model.weight_preprocessor.process_training_data(train_df)
    X, y, metadata, weights = model.create_cluster_features(processed_df, sample_weights, mode='train')
    
    if len(X) == 0:
        print("검증 데이터 부족")
        return
    
    # 동일한 데이터로 가중치 없이 학습
    uniform_weights = np.ones(len(weights))
    
    from sklearn.metrics import mean_squared_error
    from lightgbm import LGBMRegressor
    
    print(f"총 {len(X)}개 샘플로 검증")
    print(f"가중치 통계: 평균={weights.mean():.4f}, 표준편차={weights.std():.4f}")
    
    # 가중치 적용 모델
    model_weighted = LGBMRegressor(n_estimators=100, random_state=42, verbosity=-1)
    model_weighted.fit(X, y[:, 0], sample_weight=weights)  # 첫 번째 타겟만
    
    # 가중치 미적용 모델  
    model_uniform = LGBMRegressor(n_estimators=100, random_state=42, verbosity=-1)
    model_uniform.fit(X, y[:, 0], sample_weight=uniform_weights)
    
    # 예측 비교
    pred_weighted = model_weighted.predict(X)
    pred_uniform = model_uniform.predict(X)
    
    print(f"가중치 적용 MSE: {mean_squared_error(y[:, 0], pred_weighted):.4f}")
    print(f"가중치 미적용 MSE: {mean_squared_error(y[:, 0], pred_uniform):.4f}")
    print(f"예측값 차이 (평균): {abs(pred_weighted - pred_uniform).mean():.4f}")
    
    # 저가중치 샘플들의 영향 확인
    low_weight_mask = weights <= 0.1
    if low_weight_mask.sum() > 0:
        print(f"저가중치 샘플 {low_weight_mask.sum()}개의 실제 vs 예측 차이:")
        print(f"  가중치 적용: {abs(y[low_weight_mask, 0] - pred_weighted[low_weight_mask]).mean():.4f}")
        print(f"  가중치 미적용: {abs(y[low_weight_mask, 0] - pred_uniform[low_weight_mask]).mean():.4f}")

# 실행 함수
def run_cluster_based_pipeline(df:pd.DataFrame):
    """클러스터 기반 파이프라인 실행"""
    
    # 데이터 로드
    train_df = df
    train_df['date'] = pd.to_datetime(train_df['영업일자'])
    train_df[['store', 'menu']] = train_df['영업장명_메뉴명'].str.split('_', expand=True, n=1)
    train_df['sales'] = train_df['매출수량']
    train_df['month'] = train_df['date'].dt.month
    train_df['day_of_week'] = train_df['date'].dt.dayofweek
    train_df['is_weekend'] = train_df['day_of_week'].isin([5, 6])
    
    # 클러스터 기반 모델 학습
    cluster_model = ClusterBasedForecastingModel()
    cluster_model.fit(train_df)
    
    # 제출 파일 생성
    submission = pd.read_csv('sample_submission.csv')
    
    import glob
    test_files = sorted(glob.glob('TEST_*.csv'))
    
    for test_idx, test_file in enumerate(test_files):
        logging.debug(f"클러스터 기반 처리: {test_file}")
        
        test_df = pd.read_csv(test_file)
        test_df['date'] = pd.to_datetime(test_df['영업일자'])
        test_df[['store', 'menu']] = test_df['영업장명_메뉴명'].str.split('_', expand=True, n=1)
        test_df['sales'] = test_df['매출수량']
        test_df['month'] = test_df['date'].dt.month
        test_df['day_of_week'] = test_df['date'].dt.dayofweek
        test_df['is_weekend'] = test_df['day_of_week'].isin([5, 6])
        
        # 예측
        predictions, metadata = cluster_model.predict(test_df)
        
        # 제출 파일에 매핑
        test_case = f"TEST_{test_idx:02d}"
        test_rows = submission[submission['영업일자'].str.contains(test_case, na=False)].index.tolist()
        
        if len(test_rows) == 7 and len(predictions) > 0:
            numeric_cols = submission.select_dtypes(include=[np.number]).columns
            
            for day_idx, row_idx in enumerate(test_rows):
                for col_idx, col in enumerate(numeric_cols):
                    if col_idx < len(predictions):
                        pred_value = predictions[col_idx, day_idx]
                        submission.loc[row_idx, col] = max(0.0, pred_value)
    
    # submission.to_csv('./Submission/submission_3.csv', index=False)
    logging.debug("submission 생성 완료!")
    
    return submission


In [6]:

# 데이터 로드 (예시)
df = pd.read_csv('train/train.csv')
cluster_submission = run_cluster_based_pipeline(df)
cluster_submission.to_csv('./Submission/submission_9.csv', index=False, encoding='utf-8-sig')


2025-08-22 20:38:48,195 DEBUG 클러스터 기반 모델 학습 시작...


=== 가중치 기반 훈련 데이터 전처리 ===
메뉴 출시점 분석 시작...


메뉴 출시점 분석:  30%|██▉       | 57/193 [00:00<00:00, 569.64it/s]       

담하_(단체) 생목살 김치전골 2.0: 260일 출시 전
담하_(단체) 은이버섯 갈비탕: 162일 출시 전
담하_(정식) 된장찌개: 153일 출시 전
담하_(정식) 물냉면 : 153일 출시 전
담하_(정식) 비빔냉면: 153일 출시 전
담하_(후식) 물냉면: 152일 출시 전
담하_(후식) 비빔냉면: 152일 출시 전
담하_갱시기: 341일 출시 전
담하_꼬막 비빔밥: 250일 출시 전
담하_담하 한우 불고기 정식: 152일 출시 전
담하_더덕 한우 지짐: 251일 출시 전
담하_명인안동소주: 181일 출시 전
담하_명태회 비빔냉면: 152일 출시 전
담하_문막 복분자 칵테일: 254일 출시 전
담하_봉평메밀 물냉면: 152일 출시 전
라그로타_AUS (200g): 341일 출시 전
라그로타_그릴드 비프 샐러드: 250일 출시 전
라그로타_까르보나라: 341일 출시 전
라그로타_모둠 해산물 플래터: 251일 출시 전
라그로타_버섯 크림 리조또: 341일 출시 전
라그로타_시저 샐러드 : 250일 출시 전
라그로타_알리오 에 올리오 : 250일 출시 전
라그로타_양갈비 (4ps): 252일 출시 전
라그로타_한우 (200g): 342일 출시 전
라그로타_해산물 토마토 스튜 파스타: 341일 출시 전
미라시아_(오븐) 하와이안 쉬림프 피자: 251일 출시 전
미라시아_보일링 랍스타 플래터: 155일 출시 전
미라시아_보일링 랍스타 플래터(덜매운맛): 153일 출시 전
미라시아_쉬림프 투움바 파스타: 153일 출시 전
미라시아_스프라이트: 152일 출시 전


메뉴 출시점 분석: 100%|██████████| 193/193 [00:00<00:00, 1271.32it/s]     


...

클러스터 모델:  83%|████████▎ | 5/6 [00:14<00:02,  2.91s/it]      

클러스터 4 완료: brunch, 3292개 샘플, 평균 가중치: 1.000
MultiOutput 학습: 7개 타겟
가중치 통계: 평균=1.0000, 최소=1.0000, 최대=1.0000


클러스터 모델: 100%|██████████| 6/6 [00:18<00:00,  3.16s/it]      
2025-08-22 20:40:10,159 DEBUG 전체 모델 + 6개 클러스터 모델 학습 완료
2025-08-22 20:40:10,178 DEBUG 클러스터 기반 처리: TEST_00.csv
2025-08-22 20:40:10,327 DEBUG 매출 여부 예측 중...


클러스터 5 완료: other, 46626개 샘플, 평균 가중치: 1.000


예측 진행: 100%|██████████| 193/193 [00:00<00:00, 327.50it/s]
2025-08-22 20:40:11,169 DEBUG 클러스터 기반 처리: TEST_01.csv
2025-08-22 20:40:11,280 DEBUG 매출 여부 예측 중...
예측 진행: 100%|██████████| 193/193 [00:00<00:00, 329.34it/s]
2025-08-22 20:40:12,124 DEBUG 클러스터 기반 처리: TEST_02.csv
2025-08-22 20:40:12,237 DEBUG 매출 여부 예측 중...
예측 진행: 100%|██████████| 193/193 [00:00<00:00, 328.87it/s]
2025-08-22 20:40:13,084 DEBUG 클러스터 기반 처리: TEST_03.csv
2025-08-22 20:40:13,195 DEBUG 매출 여부 예측 중...
예측 진행: 100%|██████████| 193/193 [00:00<00:00, 330.70it/s]
2025-08-22 20:40:14,038 DEBUG 클러스터 기반 처리: TEST_04.csv
2025-08-22 20:40:14,150 DEBUG 매출 여부 예측 중...
예측 진행: 100%|██████████| 193/193 [00:00<00:00, 331.33it/s]
2025-08-22 20:40:14,990 DEBUG 클러스터 기반 처리: TEST_05.csv
2025-08-22 20:40:15,116 DEBUG 매출 여부 예측 중...
예측 진행: 100%|██████████| 193/193 [00:00<00:00, 281.49it/s]
2025-08-22 20:40:16,064 DEBUG 클러스터 기반 처리: TEST_06.csv
2025-08-22 20:40:16,207 DEBUG 매출 여부 예측 중...
예측 진행: 100%|██████████| 193/193 [00:00<00:00, 336.61it/s]
2025-0